# Project 1 — Large-Scale Analytics with Spark
## Notebook de Exploração e Verificação Inicial (M1, M2, M3)

Este notebook serve para prototipar, verificar esquemas de dados e testar as transformações antes de serem integradas em definitivo no script `src/main.py`.

### 1. Inicialização do Spark e Profiler

In [ ]:
import sys
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as sf
from pyspark.sql.types import *

# Adiciona o caminho src para importar o spark_profiler
sys.path.append("/workspace/src")
sys.path.append("../src")
from spark_profiler import SparkProfiler

# Define o master conforme estejas dentro do docker ou localmente
master_url = "spark://spark-master:7077" if os.path.exists("/data") else "local[*]"
data_path = "/data/" if master_url.startswith("spark://") else "../data/"

builder = (
    SparkSession.builder
    .appName("nyc-taxi-exploration")
    .master(master_url)
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "1536m")
    .config("spark.sql.adaptive.enabled", "true")
)

if master_url.startswith("spark://"):
    builder = (
        builder
        .config("spark.driver.host", "jupyter")
        .config("spark.driver.bindAddress", "0.0.0.0")
    )

spark = builder.getOrCreate()
profiler = SparkProfiler(spark)

print("Spark Session ativa!")
print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Spark UI:", spark.sparkContext.uiWebUrl)

### 2. Leitura da Tabela de Zonas (`taxi_zone_lookup.csv`)

In [ ]:
lookup_file = data_path + "taxi_zone_lookup.csv"
zones_df = spark.read.option("header", "true").option("inferSchema", "true").csv(lookup_file)

print(f"Total de zonas: {zones_df.count()}")
zones_df.show(5, truncate=False)

### 3. Leitura e Limpeza de Dados (M1)

In [ ]:
trips_file = data_path + "yellow_tripdata_2024-01.parquet"

with profiler.profile("M1_read_and_clean"):
    raw_df = spark.read.parquet(trips_file)
    
    # Condições de invalidação de acordo com o enunciado:
    # 1. fare_amount <= 0
    # 2. trip_distance <= 0
    # 3. passenger_count == 0 ou > 6
    # 4. dropoff <= pickup
    # 5. PULocationID ou DOLocationID fora de [1, 263]
    cond_valid = (
        (sf.col("fare_amount") > 0) &
        (sf.col("trip_distance") > 0) &
        (sf.col("passenger_count") > 0) & (sf.col("passenger_count") <= 6) &
        (sf.col("tpep_dropoff_datetime") > sf.col("tpep_pickup_datetime")) &
        (sf.col("PULocationID").between(1, 263)) &
        (sf.col("DOLocationID").between(1, 263))
    )
    
    clean_df = raw_df.filter(cond_valid)
    print("Total de viagens limpas:", clean_df.count())

profiler.report()

### 4. Protótipo de M2 (Análise de Procura Temporal)

In [ ]:
# Juntar com a tabela de zonas para obter o Borough de recolha (pickup_borough)
# Otimização chave: sf.broadcast(zones_df) porque a tabela de zonas tem apenas 265 linhas!
zones_small = zones_df.select(sf.col("LocationID").alias("PULocationID"), sf.col("Borough").alias("pickup_borough"))

m2_df = (
    clean_df
    .join(sf.broadcast(zones_small), on="PULocationID")
    .withColumn("day_of_week", sf.dayofweek("tpep_pickup_datetime"))
    .withColumn("hour_of_day", sf.hour("tpep_pickup_datetime"))
    .withColumn("duration_min", (sf.col("tpep_dropoff_datetime").cast("long") - sf.col("tpep_pickup_datetime").cast("long")) / 60.0)
    .groupBy("pickup_borough", "day_of_week", "hour_of_day")
    .agg(
        sf.count("*").alias("total_trips"),
        sf.round(sf.mean("fare_amount"), 2).alias("mean_fare"),
        sf.round(sf.mean("duration_min"), 2).alias("mean_duration_min"),
        sf.round(sf.mean("trip_distance"), 2).alias("mean_distance")
    )
    .orderBy("pickup_borough", "day_of_week", "hour_of_day")
)

m2_df.show(10)